In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import yaml

In [2]:
with open('../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

In [ ]:
final_demo = pl.read_csv(config["input_data"]["file1"])
final_demo.head()

client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
i64,f64,f64,f64,str,f64,f64,f64,f64
836976,6.0,73.0,60.5,"""U""",2.0,45105.3,6.0,9.0
2304905,7.0,94.0,58.0,"""U""",2.0,110860.3,6.0,9.0
1439522,5.0,64.0,32.0,"""U""",2.0,52467.79,6.0,9.0
1562045,16.0,198.0,49.0,"""M""",2.0,67454.65,3.0,6.0
5126305,12.0,145.0,33.0,"""F""",2.0,103671.75,0.0,3.0


In [4]:
experiment_data = pl.read_csv(config["input_data"]["file2"])
experiment_data.head()

client_id,Variation
i64,str
9988021,"""Test"""
8320017,"""Test"""
4033851,"""Control"""
1982004,"""Test"""
9294070,"""Control"""


In [5]:
webdata1 = pl.read_csv(config["input_data"]["file3"])
webdata1.head()

client_id,visitor_id,visit_id,process_step,date_time
i64,str,str,str,str
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:27:07"""
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_2""","""2017-04-17 15:26:51"""
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:19:22"""
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_2""","""2017-04-17 15:19:13"""
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:18:04"""


In [6]:
webdata2 = pl.read_csv(config["input_data"]["file2"])
webdata2.head()

client_id,Variation
i64,str
9988021,"""Test"""
8320017,"""Test"""
4033851,"""Control"""
1982004,"""Test"""
9294070,"""Control"""


In [7]:
webdata2["Variation"].value_counts()

Variation,count
str,u32
"""NA""",20109
"""Test""",26968
"""Control""",23532


In [8]:
webdata1["process_step"].value_counts().sort("count", descending=True)

process_step,count
str,u32
"""start""",108910
"""step_1""",73432
"""step_2""",61768
"""step_3""",53628
"""confirm""",45403


MERGED DATA

In [9]:
merged = pl.read_csv(config["output_data"]["file5"])
pl.DataFrame(merged).head()

,client_id,visitor_id,visit_id,process_step,date_time,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation
i64,i64,str,str,str,str,f64,f64,f64,str,f64,f64,f64,f64,str
0,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:27:07""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""
1,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_2""","""2017-04-17 15:26:51""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""
2,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:19:22""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""
3,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_2""","""2017-04-17 15:19:13""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""
4,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:18:04""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""


Defining SEQ steps

In [10]:
df = pl.read_csv(config["output_data"]["file5"])

STEPS = ['start', 'step_1', 'step_2', 'step_3', 'confirm']
STEP_ORDER = {step: i for step, i in zip(STEPS, range(len(STEPS)))}


In [12]:
client_visit_counts = (
    df
    .group_by(["Variation", "client_id"])
    .agg(
        pl.col("visit_id").n_unique().alias("n_visits")
    )
    .group_by(["Variation", "n_visits"])
    .agg(pl.len().alias("n_clients"))
    .sort(["Variation", "n_visits"])
)

print(client_visit_counts)

shape: (23, 3)
┌───────────┬──────────┬───────────┐
│ Variation ┆ n_visits ┆ n_clients │
│ ---       ┆ ---      ┆ ---       │
│ str       ┆ u32      ┆ u32       │
╞═══════════╪══════════╪═══════════╡
│ Control   ┆ 1        ┆ 14197     │
│ Control   ┆ 2        ┆ 2905      │
│ Control   ┆ 3        ┆ 614       │
│ Control   ┆ 4        ┆ 164       │
│ Control   ┆ 5        ┆ 56        │
│ …         ┆ …        ┆ …         │
│ Test      ┆ 6        ┆ 31        │
│ Test      ┆ 7        ┆ 8         │
│ Test      ┆ 8        ┆ 11        │
│ Test      ┆ 9        ┆ 2         │
│ Test      ┆ 10       ┆ 1         │
└───────────┴──────────┴───────────┘


Day 2, organizing thoughts

In [13]:
import polars as pl

STEP_ORDER = {"start": 0, "step_1": 1, "step_2": 2, "step_3": 3, "confirm": 4}
STEP_NAMES = {v: k for k, v in STEP_ORDER.items()}
REQUIRED_STEPS = set(STEP_ORDER.keys())
STEPS = ["start", "step_1", "step_2", "step_3", "confirm"]

df = (
    pl.read_csv(config["output_data"]["file5"])
    .with_columns([
        pl.col("date_time").str.to_datetime().alias("date_time"),
        pl.col("process_step").replace(STEP_ORDER).alias("step_rank"),
    ])
    .sort(["visit_id", "date_time"])
    .with_columns([
        pl.col("step_rank").shift(1).over("visit_id").alias("prev_step_rank"),
        pl.col("process_step").shift(1).over("visit_id").alias("prev_step_name"),
    ])
    .with_columns([
        (pl.col("step_rank") == pl.col("prev_step_rank")).alias("is_repetition"),
        (pl.col("step_rank") < pl.col("prev_step_rank")).alias("is_regression"),
    ])
    .with_columns(
        (pl.col("is_repetition") | pl.col("is_regression")).alias("is_error")
    )
)

def classify_visit(steps, step_ranks, had_error, n_regressions):
    steps = list(steps)
    step_ranks = list(step_ranks)

    # Invalid: sequence violation — a later step appears before an earlier one
    for i in range(len(STEPS) - 1):
        if STEPS[i] in steps and STEPS[i+1] in steps:
            if steps.index(STEPS[i+1]) < steps.index(STEPS[i]):
                return "invalid"

    # Incomplete: never reached confirm or missing steps
    if not REQUIRED_STEPS.issubset(set(steps)):
        return "incomplete"

    # Smooth: all steps, correct order, no errors
    if not had_error:
        return "smooth"

    # Lumpy: completed but with errors — distinguish type
    if n_regressions > 0:
        return "lumpy_regression"
    return "lumpy_repetition"

visit_agg = (
    df.group_by("visit_id").agg([
        pl.col("process_step").alias("steps_list"),
        pl.col("step_rank").alias("step_ranks_list"),
        pl.col("date_time").min().alias("visit_start"),
        pl.col("date_time").max().alias("visit_end"),
        pl.col("step_rank").max().alias("max_step_reached"),
        pl.col("is_error").any().alias("had_error"),
        pl.col("is_repetition").sum().alias("n_repetitions"),
        pl.col("is_regression").sum().alias("n_regressions"),
        pl.col("Variation").first().alias("variation"),
        pl.col("client_id").first().alias("client_id"),
        pl.col("visitor_id").first().alias("visitor_id"),
    ])
    .with_columns([
        (pl.col("visit_end") - pl.col("visit_start"))
            .dt.total_seconds()
            .alias("visit_duration_seconds"),
        pl.struct(["steps_list", "step_ranks_list", "had_error", "n_regressions"])
          .map_elements(
              lambda r: classify_visit(
                  r["steps_list"],
                  r["step_ranks_list"],
                  r["had_error"],
                  r["n_regressions"]
              ),
              return_dtype=pl.String
          )
          .alias("completion_type"),
    ])
    .with_columns(
        # dropped_at_step only meaningful for incomplete visits
        pl.when(pl.col("completion_type") == "incomplete")
            .then(pl.col("max_step_reached").cast(pl.String)
                    .replace({str(k): v for k, v in STEP_NAMES.items()}))
            .otherwise(pl.lit(None))
            .alias("dropped_at_step")
    )
)

print(
    visit_agg
    .group_by(["variation", "completion_type"])
    .agg(pl.len().alias("n_visits"))
    .sort(["variation", "completion_type"])
)

shape: (10, 3)
┌───────────┬──────────────────┬──────────┐
│ variation ┆ completion_type  ┆ n_visits │
│ ---       ┆ ---              ┆ ---      │
│ str       ┆ str              ┆ u32      │
╞═══════════╪══════════════════╪══════════╡
│ Control   ┆ incomplete       ┆ 12334    │
│ Control   ┆ invalid          ┆ 65       │
│ Control   ┆ lumpy_regression ┆ 2583     │
│ Control   ┆ lumpy_repetition ┆ 1250     │
│ Control   ┆ smooth           ┆ 7037     │
│ Test      ┆ incomplete       ┆ 14363    │
│ Test      ┆ invalid          ┆ 125      │
│ Test      ┆ lumpy_regression ┆ 3403     │
│ Test      ┆ lumpy_repetition ┆ 2507     │
│ Test      ┆ smooth           ┆ 8331     │
└───────────┴──────────────────┴──────────┘


For lumpy the rule was:

- All 5 steps must appear
- When you take only the first occurrence of each step, they must be in the correct order (start → 1 → 2 → 3 → confirm)
- Repetitions and regressions are allowed in between

So start-1-2-1-2-3-confirm IS OK but start-2-1-3-confirm IS NOT (first occurrence of 2 comes before 1).

Finding out completion in one visit (smooth + lumpy, by variation)

Completed on first visit (first visit per client)

In [14]:
# Visits that completed start->confirm in a single visit_id (smooth or lumpy, no matter)
completed_first_visit = (
    visit_agg
    .filter(
        pl.col("completion_type").is_in(["smooth", "lumpy_regression", "lumpy_repetition"])
    )
)

# How many clients had exactly one visit_id and completed it
single_visit_completions = (
    completed_first_visit
    .group_by("client_id")
    .agg(pl.len().alias("n_visits"))
    .filter(pl.col("n_visits") == 1)
)

total_clients = visit_agg["client_id"].n_unique()
completed_count = len(single_visit_completions)

print(f"Total unique clients:                {total_clients}")
print(f"Completed in one visit (any):        {completed_count}")
print(f"Rate:                                {completed_count / total_clients:.2%}")
print()
print("--- By Variation ---")
print(
    completed_first_visit
    .join(single_visit_completions.select("client_id"), on="client_id")
    .group_by(["variation", "completion_type"])
    .agg(pl.len().alias("n_clients"))
    .sort(["variation", "completion_type"])
)

Total unique clients:                39870
Completed in one visit (any):        25046
Rate:                                62.82%

--- By Variation ---
shape: (6, 3)
┌───────────┬──────────────────┬───────────┐
│ variation ┆ completion_type  ┆ n_clients │
│ ---       ┆ ---              ┆ ---       │
│ str       ┆ str              ┆ u32       │
╞═══════════╪══════════════════╪═══════════╡
│ Control   ┆ lumpy_regression ┆ 2575      │
│ Control   ┆ lumpy_repetition ┆ 1246      │
│ Control   ┆ smooth           ┆ 7023      │
│ Test      ┆ lumpy_regression ┆ 3392      │
│ Test      ┆ lumpy_repetition ┆ 2505      │
│ Test      ┆ smooth           ┆ 8305      │
└───────────┴──────────────────┴───────────┘


Errors per step transition

In [15]:
# Errors per step transition — ranked buggiest to least buggy per group. Added ERROR RATE.
 
transition_errors = (
    df
    .filter(pl.col("is_error"))
    .with_columns(
        (pl.col("prev_step_name") + " -> " + pl.col("process_step")).alias("transition")
    )
    .group_by(["Variation", "transition"])
    .agg([
        pl.len().alias("n_errors"),
        pl.col("is_repetition").sum().alias("n_repetitions"),
        pl.col("is_regression").sum().alias("n_regressions"),
    ])
    .join(
        df.group_by("Variation").agg(pl.len().alias("total_rows")),
        on="Variation"
    )
    .with_columns(
        (pl.col("n_errors") / pl.col("total_rows")).alias("error_rate")
    )
    .sort(["Variation", "n_errors"], descending=[False, True])
)

# Buggiest transition per group
buggiest_per_group = (
    transition_errors
    .group_by("Variation")
    .agg(pl.all().sort_by("n_errors", descending=True).first())
    .sort("Variation")
)

print("=== Buggiest Transitions by Variation ===")
print(transition_errors)

print("\n=== Buggiest Transition Per Group ===")
print(buggiest_per_group.select(["Variation", "transition", "n_errors", "n_repetitions", "n_regressions", "error_rate"]))

=== Buggiest Transitions by Variation ===
shape: (29, 7)
┌───────────┬─────────────────┬──────────┬───────────────┬───────────────┬────────────┬────────────┐
│ Variation ┆ transition      ┆ n_errors ┆ n_repetitions ┆ n_regressions ┆ total_rows ┆ error_rate │
│ ---       ┆ ---             ┆ ---      ┆ ---           ┆ ---           ┆ ---        ┆ ---        │
│ str       ┆ str             ┆ u32      ┆ u32           ┆ u32           ┆ u32        ┆ f64        │
╞═══════════╪═════════════════╪══════════╪═══════════════╪═══════════════╪════════════╪════════════╡
│ Control   ┆ start -> start  ┆ 6118     ┆ 6118          ┆ 0             ┆ 100601     ┆ 0.060815   │
│ Control   ┆ step_3 ->       ┆ 1946     ┆ 0             ┆ 1946          ┆ 100601     ┆ 0.019344   │
│           ┆ step_2          ┆          ┆               ┆               ┆            ┆            │
│ Control   ┆ step_1 -> start ┆ 1152     ┆ 0             ┆ 1152          ┆ 100601     ┆ 0.011451   │
│ Control   ┆ step_1 ->       ┆ 84

Completion Rate vs Error Rate — Test vs Control.
Trying to understand out of the completed, how many had errors. Meaning, out of COMPLETED (no incomplete, no invalid) how many were lumpy.


In [16]:
# Completion Rate vs Error Rate — Test vs Control
# Out of completed visits only, how many were smooth vs lumpy

completed_only = (
    visit_agg
    .filter(
        pl.col("completion_type").is_in(["smooth", "lumpy_regression", "lumpy_repetition"])
    )
)

error_within_completed = (
    completed_only
    .group_by(["variation", "completion_type"])
    .agg(pl.len().alias("n_visits"))
    .join(
        completed_only.group_by("variation").agg(pl.len().alias("total_completed")),
        on="variation"
    )
    .with_columns(
        (pl.col("n_visits") / pl.col("total_completed")).alias("rate")
    )
    .sort(["variation", "completion_type"])
)

print("=== Within Completed Visits: Smooth vs Lumpy by Variation ===")
print(error_within_completed)

# --- Conclusion ---
for variation in ["Test", "Control"]:
    group = error_within_completed.filter(pl.col("variation") == variation)
    smooth_rate = group.filter(pl.col("completion_type") == "smooth")["rate"][0]
    lumpy_rate = 1 - smooth_rate
    print(f"\n{variation}: {smooth_rate:.1%} of completions were smooth, {lumpy_rate:.1%} encountered errors along the way.")

=== Within Completed Visits: Smooth vs Lumpy by Variation ===
shape: (6, 5)
┌───────────┬──────────────────┬──────────┬─────────────────┬──────────┐
│ variation ┆ completion_type  ┆ n_visits ┆ total_completed ┆ rate     │
│ ---       ┆ ---              ┆ ---      ┆ ---             ┆ ---      │
│ str       ┆ str              ┆ u32      ┆ u32             ┆ f64      │
╞═══════════╪══════════════════╪══════════╪═════════════════╪══════════╡
│ Control   ┆ lumpy_regression ┆ 2583     ┆ 10870           ┆ 0.237626 │
│ Control   ┆ lumpy_repetition ┆ 1250     ┆ 10870           ┆ 0.114995 │
│ Control   ┆ smooth           ┆ 7037     ┆ 10870           ┆ 0.647378 │
│ Test      ┆ lumpy_regression ┆ 3403     ┆ 14241           ┆ 0.238958 │
│ Test      ┆ lumpy_repetition ┆ 2507     ┆ 14241           ┆ 0.176041 │
│ Test      ┆ smooth           ┆ 8331     ┆ 14241           ┆ 0.585001 │
└───────────┴──────────────────┴──────────┴─────────────────┴──────────┘

Test: 58.5% of completions were smooth, 41.5% e

Completion rate = (smooth + lumpy visits) - anyone who reached confirm regardless of errors
vs
Incompletion Rate (incomplete + invalid) - anyone who did NOT finish process

In [17]:
# Completion Rate vs Error Rate — Test vs Control

completed_only = (
    visit_agg
    .filter(
        pl.col("completion_type").is_in(["smooth", "lumpy_regression", "lumpy_repetition"])
    )
)

error_within_completed = (
    completed_only
    .group_by(["variation", "completion_type"])
    .agg(pl.len().alias("n_visits"))
    .join(
        completed_only.group_by("variation").agg(pl.len().alias("total_completed")),
        on="variation"
    )
    .with_columns(
        (pl.col("n_visits") / pl.col("total_completed")).alias("rate")
    )
    .sort(["variation", "completion_type"])
)

completion_vs_not = (
    visit_agg
    .group_by(["variation", "completion_type"])
    .agg(pl.len().alias("n_visits"))
    .join(
        visit_agg.group_by("variation").agg(pl.len().alias("total_visits")),
        on="variation"
    )
    .with_columns(
        (pl.col("n_visits") / pl.col("total_visits")).alias("rate")
    )
    .sort(["variation", "completion_type"])
)

test_rate = completion_vs_not.filter(
    (pl.col("variation") == "Test") &
    pl.col("completion_type").is_in(["smooth", "lumpy_regression", "lumpy_repetition"])
)["rate"].sum()

control_rate = completion_vs_not.filter(
    (pl.col("variation") == "Control") &
    pl.col("completion_type").is_in(["smooth", "lumpy_regression", "lumpy_repetition"])
)["rate"].sum()

diff = test_rate - control_rate

print("=== Within Completed Visits: Smooth vs Lumpy by Variation ===")
print(error_within_completed)

print("\n=== Full Breakdown by Variation ===")
print(completion_vs_not)

print("\n=== Conclusion ===")
for variation, rate in [("Test", test_rate), ("Control", control_rate)]:
    group = completion_vs_not.filter(pl.col("variation") == variation)
    rates = {row["completion_type"]: row["rate"] for row in group.iter_rows(named=True)}
    incomplete = rates.get("incomplete", 0)
    invalid = rates.get("invalid", 0)
    print(f"{variation}: {rate:.1%} completed | {incomplete:.1%} incomplete | {invalid:.1%} invalid")

print(f"\nDifference (Test - Control): {diff:+.1%} — {'Test performs better' if diff > 0 else 'Control performs better'}")

=== Within Completed Visits: Smooth vs Lumpy by Variation ===
shape: (6, 5)
┌───────────┬──────────────────┬──────────┬─────────────────┬──────────┐
│ variation ┆ completion_type  ┆ n_visits ┆ total_completed ┆ rate     │
│ ---       ┆ ---              ┆ ---      ┆ ---             ┆ ---      │
│ str       ┆ str              ┆ u32      ┆ u32             ┆ f64      │
╞═══════════╪══════════════════╪══════════╪═════════════════╪══════════╡
│ Control   ┆ lumpy_regression ┆ 2583     ┆ 10870           ┆ 0.237626 │
│ Control   ┆ lumpy_repetition ┆ 1250     ┆ 10870           ┆ 0.114995 │
│ Control   ┆ smooth           ┆ 7037     ┆ 10870           ┆ 0.647378 │
│ Test      ┆ lumpy_regression ┆ 3403     ┆ 14241           ┆ 0.238958 │
│ Test      ┆ lumpy_repetition ┆ 2507     ┆ 14241           ┆ 0.176041 │
│ Test      ┆ smooth           ┆ 8331     ┆ 14241           ┆ 0.585001 │
└───────────┴──────────────────┴──────────┴─────────────────┴──────────┘

=== Full Breakdown by Variation ===
shape: (10,

Trying to spot differences by age/gender. EDIT: GENDER COLUMN NEEDS TO BE CLEAN

# Building demographics

In [23]:
client_demographics = (
    df
    .select(["client_id", "clnt_age"])
    .unique(subset=["client_id"])  # one row per client
    .rename({"clnt_age": "age"})
    .with_columns(
        pl.when(pl.col("age") <= 30).then(pl.lit("18-30"))
          .when(pl.col("age") <= 45).then(pl.lit("31-45"))
          .when(pl.col("age") <= 60).then(pl.lit("46-60"))
          .otherwise(pl.lit("60+"))
          .alias("age_group")
    )
)

visit_agg_demo = visit_agg.join(client_demographics, on="client_id", how="left")
print(visit_agg_demo.select(["client_id", "age", "age_group"]).head(5))

shape: (5, 3)
┌───────────┬──────┬───────────┐
│ client_id ┆ age  ┆ age_group │
│ ---       ┆ ---  ┆ ---       │
│ i64       ┆ f64  ┆ str       │
╞═══════════╪══════╪═══════════╡
│ 3561384   ┆ 59.5 ┆ 46-60     │
│ 7338123   ┆ 23.5 ┆ 18-30     │
│ 105007    ┆ 35.0 ┆ 31-45     │
│ 5623007   ┆ 78.0 ┆ 60+       │
│ 4823947   ┆ 52.0 ┆ 46-60     │
└───────────┴──────┴───────────┘


Error rate by age_group

In [24]:
error_rate_demo = (
    visit_agg_demo
    .group_by(["variation", "age_group"])
    .agg([
        pl.len().alias("total_visits"),
        pl.col("had_error").cast(pl.Int32).sum().alias("visits_with_errors"),
    ])
    .with_columns(
        (pl.col("visits_with_errors") / pl.col("total_visits") * 100)
          .round(2).alias("error_rate_%")
    )
    .sort(["variation", "age_group"])
)
print(error_rate_demo)

shape: (8, 5)
┌───────────┬───────────┬──────────────┬────────────────────┬──────────────┐
│ variation ┆ age_group ┆ total_visits ┆ visits_with_errors ┆ error_rate_% │
│ ---       ┆ ---       ┆ ---          ┆ ---                ┆ ---          │
│ str       ┆ str       ┆ u32          ┆ i32                ┆ f64          │
╞═══════════╪═══════════╪══════════════╪════════════════════╪══════════════╡
│ Control   ┆ 18-30     ┆ 3834         ┆ 1309               ┆ 34.14        │
│ Control   ┆ 31-45     ┆ 5803         ┆ 1820               ┆ 31.36        │
│ Control   ┆ 46-60     ┆ 7186         ┆ 2539               ┆ 35.33        │
│ Control   ┆ 60+       ┆ 6446         ┆ 2279               ┆ 35.36        │
│ Test      ┆ 18-30     ┆ 4703         ┆ 1896               ┆ 40.31        │
│ Test      ┆ 31-45     ┆ 7204         ┆ 2900               ┆ 40.26        │
│ Test      ┆ 46-60     ┆ 9014         ┆ 4313               ┆ 47.85        │
│ Test      ┆ 60+       ┆ 7808         ┆ 3817               ┆ 

Completion rate by gender & age

Smooth vs Lumpy by gender & age